In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
from pathlib import Path
from solve_kis import solve_kis

from siglip_encoder import SiglipEncoder
from search_clip import SiglipRetriever 
from caption_retriever import CaptionRetriever
from paths import KIS_ROOT, DATA_DIR


In [ ]:
from bm25_retriever import BM25Retriever

In [ ]:
encoder = SiglipEncoder()
retriever = SiglipRetriever()

print("SigLIP OK")

In [ ]:
print("[*] Đang khởi tạo Caption Retriever (toàn bộ dữ liệu)...")

try:
    caption_retriever = CaptionRetriever()
    
    # Thống kê nhanh số video có trong mapping để dễ kiểm tra dữ liệu.
    unique_videos = sorted(
        {item["video_id"] for item in caption_retriever.mapping}
    ) if caption_retriever.mapping else []

    print(
        f"[*] caption_retriever sẵn sàng: "
        f"{caption_retriever.index.ntotal} vectors, "
        f"{len(unique_videos)} videos"
    )

except FileNotFoundError as e:
    caption_retriever = None
    print(f"[Lỗi] Không tìm thấy dữ liệu caption/index/mapping: {e}")
    print(
        "    Hãy đảm bảo đã có file trong các thư mục:\n"
        "    - caption_generator/Lxx/*.json\n"
        "    - index/Lxx/*.index\n"
        "    - caption_mapping/Lxx/*.json"
    )

In [ ]:
all_captions = caption_retriever.caption_data
print(f"[*] Đã thu thập được {len(all_captions)} frames caption để nạp vào BM25.")



In [ ]:
from object_filter import ObjectMetadataLookup
object_lookup = ObjectMetadataLookup(
    data_dir=DATA_DIR
)

print("Object Lookup OK")

In [ ]:
bm25_engine = BM25Retriever(all_captions)

In [ ]:
from vlm_reranker import VLMReranker

# Khởi tạo mô hình chấm điểm (Quá trình này tốn vài phút để tải checkpoint)
print("[*] Đang nạp Qwen2.5-VL Reranker lên GPU...")
vlm_engine = VLMReranker()
print("[*] VLM đã sẵn sàng!")

In [ ]:
import json

with open(KIS_ROOT / "test.json", "r", encoding="utf-8") as f:
    structured_query = json.load(f)

print(json.dumps(structured_query, ensure_ascii=False, indent=2))

In [ ]:
print(f"\n[*] Bắt đầu giải quyết query: {structured_query.get('query_id')}")
final_ranked = solve_kis(
    structured_query=structured_query,
    visual_retriever=retriever,
    visual_encoder=encoder,
    text_bm25_retriever=bm25_engine,
    text_semantic_retriever=caption_retriever,
    object_lookup=object_lookup,
    vlm_reranker=vlm_engine,
    retrieval_top_k=100,
    rerank_top_k=20,
    result_top_k=100,
)

print("\n=== TOP 10 KẾT QUẢ CUỐI ===")
for i, result in enumerate(final_ranked[:10], 1):
    print(
        f"{i:02d}. video={result['video_id']} "
        f"frame={result['frame_id']} "
        f"score={result['final_score']:.4f}"
    )

In [ ]:
csv_output_dir = Path.home() / "Downloads"
if not csv_output_dir.is_dir():
    csv_output_dir = Path.home()

csv_output_path = csv_output_dir / "query-p2-36-kis.csv"

with csv_output_path.open(
    "w",
    encoding="utf-8",
    newline=""
 ) as csv_file:
    for result in final_ranked[:100]:
        csv_file.write(
            f"{result['video_id']}, {result['frame_id']}\n"
        )

